
# Silver Layer Pipeline - Fraud Detection
**Purpose**: Clean, standardize, and enrich bronze data

**Transformations**:
- Clean data types (remove $ signs, parse dates)
- Join fraud labels to transactions
- Join MCC descriptions to transactions
- Derive temporal features (day of week, hour, month)
- Handle nulls and missing values
- Create consistent naming conventions

**Tables Created**:
- silver_transactions (enriched with fraud labels and MCC descriptions)
- silver_cards (cleaned)
- silver_users (cleaned)


In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

# Configuration
BRONZE_PATH = "databricks_workspace.bronze"
SILVER_PATH = "databricks_workspace.silver"


## 1. Silver Transactions (Main Enriched Table)

In [0]:
# Read bronze transactions
df_transactions = spark.table(f"{BRONZE_PATH}.transactions")
df_fraud_labels = spark.table(f"{BRONZE_PATH}.fraud_labels")
df_mcc_codes = spark.table(f"{BRONZE_PATH}.mcc_codes")

# Clean transactions data
df_transactions_cleaned = (df_transactions
    # Clean amount - remove $ and convert to decimal
    .withColumn("amount_clean", regexp_replace(col("amount"), "\\$|,", ""))
    .withColumn("amount_clean", col("amount_clean").cast("decimal(10,2)"))
    
    # Parse date and extract temporal features
    .withColumn("transaction_timestamp", col("date").cast("timestamp"))
    .withColumn("transaction_date", to_date(col("transaction_timestamp")))
    .withColumn("transaction_year", year(col("transaction_timestamp")))
    .withColumn("transaction_month", month(col("transaction_timestamp")))
    .withColumn("transaction_day", dayofmonth(col("transaction_timestamp")))
    .withColumn("transaction_hour", hour(col("transaction_timestamp")))
    .withColumn("transaction_dayofweek", dayofweek(col("transaction_timestamp")))  # 1=Sunday, 7=Saturday
    .withColumn("transaction_dayname", date_format(col("transaction_timestamp"), "EEEE"))
    
    # Create time of day categories
    .withColumn("time_of_day", 
        when((col("transaction_hour") >= 6) & (col("transaction_hour") < 12), "Morning")
        .when((col("transaction_hour") >= 12) & (col("transaction_hour") < 18), "Afternoon")
        .when((col("transaction_hour") >= 18) & (col("transaction_hour") < 22), "Evening")
        .otherwise("Night")
    )
    
    # Week number for weekly aggregations
    .withColumn("transaction_week", weekofyear(col("transaction_timestamp")))
    
    # Standardize column names
    .withColumnRenamed("id", "transaction_id")
    .withColumnRenamed("use_chip", "transaction_method")
    .withColumnRenamed("mcc", "mcc_code")
)

In [0]:
# Join with fraud labels
df_with_fraud = (df_transactions_cleaned
    .join(df_fraud_labels, 
          df_transactions_cleaned.transaction_id == df_fraud_labels.transaction_id,
          "left")
    .drop(df_fraud_labels.transaction_id)
    
    # Create binary fraud flag
    .withColumn("is_fraud", 
        when(col("fraud_label") == "Yes", True)
        .when(col("fraud_label") == "No", False)
        .otherwise(None)
    )
)

In [0]:
# Join with MCC codes for merchant category descriptions
df_silver_transactions = (df_with_fraud
    .join(df_mcc_codes,
          df_with_fraud.mcc_code == df_mcc_codes.mcc_code,
          "left")
    .drop(df_mcc_codes.mcc_code)
    
    # Handle missing MCC descriptions
    .withColumn("mcc_description", 
        when(col("mcc_description").isNull(), "Unknown Category")
        .otherwise(col("mcc_description"))
    )
)

In [0]:
#Select and order final columns
df_silver_transactions_final = df_silver_transactions.select(
    # Identifiers
    "transaction_id",
    "client_id",
    "card_id",
    "merchant_id",
    
    # Transaction details
    "amount_clean",
    "transaction_method",
    "mcc_code",
    "mcc_description",
    
    # Merchant details
    "merchant_city",
    "merchant_state",
    "zip",
    
    # Temporal fields
    "transaction_timestamp",
    "transaction_date",
    "transaction_year",
    "transaction_month",
    "transaction_day",
    "transaction_hour",
    "transaction_dayofweek",
    "transaction_dayname",
    "transaction_week",
    "time_of_day",
    
    # Fraud indicators
    "is_fraud",
    "fraud_label",
    
    # Metadata
    "errors",
)

In [0]:
# Write to Delta
(df_silver_transactions_final.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_PATH}.transactions")
)

print(f"Silver transactions created: {df_silver_transactions_final.count():,} rows")
display(df_silver_transactions_final.limit(10))

Silver transactions created: 13,305,915 rows


transaction_id,client_id,card_id,merchant_id,amount_clean,transaction_method,mcc_code,mcc_description,merchant_city,merchant_state,zip,transaction_timestamp,transaction_date,transaction_year,transaction_month,transaction_day,transaction_hour,transaction_dayofweek,transaction_dayname,transaction_week,time_of_day,is_fraud,fraud_label,errors
14358050,997,5825,14528,1.46,Swipe Transaction,5499,Miscellaneous Food Stores,Chula Vista,CA,91913.0,2014-05-03T11:58:00.000Z,2014-05-03,2014,5,3,11,7,Saturday,18,Morning,false,No,null
7475332,848,3915,13051,46.41,Swipe Transaction,5813,Drinking Places (Alcoholic Beverages),Harwood,MD,20776.0,2010-01-01T00:06:00.000Z,2010-01-01,2010,1,1,0,6,Friday,53,Night,false,No,null
9190171,1024,1006,41904,16.08,Swipe Transaction,5813,Drinking Places (Alcoholic Beverages),Andover,KS,67002.0,2011-02-22T21:49:00.000Z,2011-02-22,2011,2,22,21,3,Tuesday,8,Evening,false,No,null
19585526,1719,4942,35451,4.64,Chip Transaction,5812,Eating Places and Restaurants,Miami,FL,33183.0,2017-05-30T12:03:00.000Z,2017-05-30,2017,5,30,12,3,Tuesday,22,Afternoon,false,No,null
10916269,1433,5841,69972,17.80,Swipe Transaction,5814,Fast Food Restaurants,Vicksburg,MS,39180.0,2012-03-28T11:12:00.000Z,2012-03-28,2012,3,28,11,4,Wednesday,13,Morning,null,null,null
9190169,1654,1200,60569,81.75,Swipe Transaction,5300,Wholesale Clubs,Lincoln,RI,2865.0,2011-02-22T21:48:00.000Z,2011-02-22,2011,2,22,21,3,Tuesday,8,Evening,null,null,null
21338896,581,2320,60569,17.86,Chip Transaction,5300,Wholesale Clubs,Meraux,LA,70075.0,2018-06-05T22:03:00.000Z,2018-06-05,2018,6,5,22,3,Tuesday,23,Night,null,null,null
21338886,1775,344,73844,73.63,Chip Transaction,5812,Eating Places and Restaurants,Milford,DE,19963.0,2018-06-05T21:58:00.000Z,2018-06-05,2018,6,5,21,3,Tuesday,23,Evening,null,null,null
21338895,94,2890,27092,80.00,Chip Transaction,4829,Money Transfer,Morganton,NC,28680.0,2018-06-05T22:03:00.000Z,2018-06-05,2018,6,5,22,3,Tuesday,23,Night,null,null,null
17838215,571,312,46965,51.98,Chip Transaction,7538,Automotive Service Shops,Martinsburg,WV,25401.0,2016-05-23T13:01:00.000Z,2016-05-23,2016,5,23,13,2,Monday,21,Afternoon,null,null,null


## 2. Silver Cards

In [0]:
df_cards = spark.table(f"{BRONZE_PATH}.cards")

df_silver_cards = (df_cards
    # Clean credit limit - remove $ and convert to decimal
    .withColumn("credit_limit_clean", regexp_replace(col("credit_limit"), "\\$|,", ""))
    .withColumn("credit_limit_clean", col("credit_limit_clean").cast("decimal(10,2)"))
    
    # Parse dates
    .withColumn("acct_open_date_parsed", to_date(col("acct_open_date"), "MM/yyyy"))
    .withColumn("expires_parsed", to_date(col("expires"), "MM/yyyy"))
    
    # Standardize Yes/No to boolean
    .withColumn("has_chip_flag", when(upper(col("has_chip")) == "YES", True).otherwise(False))
    .withColumn("card_on_dark_web_flag", when(upper(col("card_on_dark_web")) == "YES", True).otherwise(False))
    
    # Standardize column names
    .withColumnRenamed("id", "card_id")
    
    # Select final columns
    .select(
        "card_id",
        "client_id",
        "card_brand",
        "card_type",
        "card_number",
        "expires_parsed",
        "cvv",
        "has_chip_flag",
        "num_cards_issued",
        "credit_limit_clean",
        "acct_open_date_parsed",
        "year_pin_last_changed",
        "card_on_dark_web_flag",
    )
)

# Write to Delta
(df_silver_cards.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_PATH}.cards")
)

print(f"Silver cards created: {df_silver_cards.count():,} rows")
display(df_silver_cards.limit(10))

✓ Silver cards created: 6,146 rows


card_id,client_id,card_brand,card_type,card_number,expires_parsed,cvv,has_chip_flag,num_cards_issued,credit_limit_clean,acct_open_date_parsed,year_pin_last_changed,card_on_dark_web_flag
4524,825,Visa,Debit,4344676511950444,2022-12-01,623,true,2,24295.00,2002-09-01,2008,false
2731,825,Visa,Debit,4956965974959986,2020-12-01,393,true,2,21968.00,2014-04-01,2014,false
3701,825,Visa,Debit,4582313478255491,2024-02-01,719,true,2,46414.00,2003-07-01,2004,false
42,825,Visa,Credit,4879494103069057,2024-08-01,693,false,1,12400.00,2003-01-01,2012,false
4659,825,Mastercard,Debit (Prepaid),5722874738736011,2009-03-01,75,true,1,28.00,2008-09-01,2009,false
4537,1746,Visa,Credit,4404898874682993,2003-09-01,736,true,1,27500.00,2003-09-01,2012,false
1278,1746,Visa,Debit,4001482973848631,2022-07-01,972,true,2,28508.00,2011-02-01,2011,false
3687,1746,Mastercard,Debit,5627220683410948,2022-06-01,48,true,2,9022.00,2003-07-01,2015,false
3465,1746,Mastercard,Debit (Prepaid),5711382187309326,2020-11-01,722,true,2,54.00,2010-06-01,2015,false
3754,1746,Mastercard,Debit (Prepaid),5766121508358701,2023-02-01,908,true,1,99.00,2006-07-01,2012,false



## 3. Silver Users

In [0]:
df_users = spark.table(f"{BRONZE_PATH}.users")

df_silver_users = (df_users
    # Clean monetary fields - remove $ and convert to decimal
    .withColumn("per_capita_income_clean", regexp_replace(col("per_capita_income"), "\\$|,", ""))
    .withColumn("per_capita_income_clean", col("per_capita_income_clean").cast("decimal(10,2)"))
    
    .withColumn("yearly_income_clean", regexp_replace(col("yearly_income"), "\\$|,", ""))
    .withColumn("yearly_income_clean", col("yearly_income_clean").cast("decimal(10,2)"))
    
    .withColumn("total_debt_clean", regexp_replace(col("total_debt"), "\\$|,", ""))
    .withColumn("total_debt_clean", col("total_debt_clean").cast("decimal(10,2)"))
    
    # Calculate debt-to-income ratio
    .withColumn("debt_to_income_ratio",
        when(col("yearly_income_clean") > 0,
             col("total_debt_clean") / col("yearly_income_clean")
        ).otherwise(None)
    )
    
    # Standardize column names
    .withColumnRenamed("id", "user_id")
    
    # Select final columns
    .select(
        "user_id",
        "current_age",
        "retirement_age",
        "birth_year",
        "birth_month",
        "gender",
        "address",
        "latitude",
        "longitude",
        "per_capita_income_clean",
        "yearly_income_clean",
        "total_debt_clean",
        "debt_to_income_ratio",
        "credit_score",
        "num_credit_cards",
    )
)

# Write to Delta
(df_silver_users.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{SILVER_PATH}.users")
)

print(f"Silver users created: {df_silver_users.count():,} rows")
display(df_silver_users.limit(10))

Silver users created: 2,000 rows


user_id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income_clean,yearly_income_clean,total_debt_clean,debt_to_income_ratio,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,29278.00,59696.00,127613.00,2.1377144197266,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,37891.00,77254.00,191349.00,2.4768814559764,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,22681.00,33483.00,196.00,0.0058537168115,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,163145.00,249925.00,202328.00,0.8095548664599,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,53797.00,109687.00,183855.00,1.6761785808710,675,1
68,42,70,1977,10,Male,58 Birch Lane,41.55,-90.6,20599.00,41997.00,0.00,0E-13,704,3
1075,36,67,1983,12,Female,5695 Fifth Street,38.22,-85.74,25258.00,51500.00,102286.00,1.9861359223301,672,3
1711,26,67,1993,12,Male,1941 Ninth Street,45.51,-122.64,26790.00,54623.00,114711.00,2.1000494297274,728,1
1116,81,66,1938,7,Female,11 Spruce Avenue,40.32,-75.32,26273.00,42509.00,2895.00,0.0681032251994,755,5
1752,34,60,1986,1,Female,887 Grant Street,29.97,-92.12,18730.00,38190.00,81262.00,2.1278345116523,810,1


## 4. Validation and Summary

In [0]:
print("=" * 80)
print("SILVER LAYER VALIDATION SUMMARY")
print("=" * 80)

# Transactions validation
trans_count = spark.table(f"{SILVER_PATH}.transactions").count()
fraud_count = spark.table(f"{SILVER_PATH}.transactions").filter(col("is_fraud") == True).count()
fraud_pct = (fraud_count / trans_count * 100) if trans_count > 0 else 0

print(f"\n✓ Silver Transactions: {trans_count:,} rows")
print(f"  - Fraudulent: {fraud_count:,} ({fraud_pct:.2f}%)")
print(f"  - Legitimate: {trans_count - fraud_count:,} ({100-fraud_pct:.2f}%)")

# Cards validation
cards_count = spark.table(f"{SILVER_PATH}.cards").count()
print(f"\n✓ Silver Cards: {cards_count:,} rows")

# Users validation
users_count = spark.table(f"{SILVER_PATH}.users").count()
print(f"\n✓ Silver Users: {users_count:,} rows")

print("\n" + "=" * 80)
print("Silver layer pipeline completed successfully!")
print("=" * 80)

SILVER LAYER VALIDATION SUMMARY

✓ Silver Transactions: 13,305,915 rows
  - Fraudulent: 13,332 (0.10%)
  - Legitimate: 13,292,583 (99.90%)

✓ Silver Cards: 6,146 rows

✓ Silver Users: 2,000 rows

Silver layer pipeline completed successfully!


## 5. Data Quality Checks

In [0]:
from pyspark.sql.functions import col, min, max

# Load tables once (clean + efficient)
transactions_df = spark.table(f"{SILVER_PATH}.transactions").alias("t")
cards_df = spark.table(f"{SILVER_PATH}.cards").alias("c")
users_df = spark.table(f"{SILVER_PATH}.users").alias("u")

# Optional: total transaction count (used in MCC coverage)
trans_count = transactions_df.count()

# --------------------------------------------------
# Check for null amounts
# --------------------------------------------------
null_amounts = (
    transactions_df
    .filter(col("amount_clean").isNull())
    .count()
)

print(f"Transactions with null amounts: {null_amounts}")

# --------------------------------------------------
# Check transaction date range
# --------------------------------------------------
date_range = (
    transactions_df
    .select(
        min("transaction_date").alias("earliest"),
        max("transaction_date").alias("latest")
    )
    .first()
)

print(f"\nTransaction date range: {date_range.earliest} to {date_range.latest}")

# --------------------------------------------------
# MCC coverage
# --------------------------------------------------
mcc_coverage = (
    transactions_df
    .filter(col("mcc_description") != "Unknown Category")
    .count() / trans_count * 100
)

print(f"\nMCC code coverage: {mcc_coverage:.2f}%")

# --------------------------------------------------
# Check for orphaned card records
# --------------------------------------------------
orphaned_cards = (
    transactions_df
    .join(
        cards_df,
        col("t.card_id") == col("c.card_id"),
        "left_anti"
    )
    .count()
)

print(f"\nTransactions with missing card info: {orphaned_cards}")

# --------------------------------------------------
# Check for orphaned user records
# --------------------------------------------------
orphaned_users = (
    transactions_df
    .join(
        users_df,
        col("t.client_id") == col("u.user_id"),
        "left_anti"
    )
    .count()
)

print(f"Transactions with missing user info: {orphaned_users}")


Transactions with null amounts: 0

Transaction date range: 2010-01-01 to 2019-10-31

MCC code coverage: 100.00%

Transactions with missing card info: 0
Transactions with missing user info: 0
